# Data Packing — Pre-Pack Episodes for Maximum Training Throughput

Training from compressed video is bottlenecked by **random I/O**: opening different files, seeking to random positions, decoding throwaway frames.

**Data packing** extracts clips offline, shuffles across episodes, and writes them sequentially into shard videos with GOP=1. Training reads front-to-back — no seeking, no wasted decodes.

This notebook:
1. Generates a realistic synthetic dataset (100 episodes, 720p, GOP=30)
2. Packs into shards and inspects the manifest
3. **Proves the speedup** — isolates file-open cost, seek cost, and sequential read advantage
4. Advanced: overlapping clips, codec comparison, versioned re-packing
5. Full training pipeline with packed shards

> **LeRobot note**: LeRobot v2.0+ already concatenates episodes into single MP4s per chunk — they independently arrived at the same insight. frameforge's packing goes further with cross-episode shuffling, configurable clip extraction, versioning, and codec choice.

---
## 1. Generate Synthetic Dataset (100 episodes, 720p, GOP=30)

In [ ]:
import av, numpy as np, time, random, shutil, json
from pathlib import Path

from loguru import logger
logger.disable('frameforge')

WORK_DIR = Path('_packing_demo')
if WORK_DIR.exists(): shutil.rmtree(WORK_DIR)
EPISODE_DIR = WORK_DIR / 'episodes'
EPISODE_DIR.mkdir(parents=True)

NUM_EPISODES = 100
FPS, WIDTH, HEIGHT = 30, 1280, 720

def make_episode(path, num_frames, idx, gop=30):
    c = av.open(str(path), mode='w')
    s = c.add_stream('h264', rate=FPS)
    s.width, s.height, s.pix_fmt = WIDTH, HEIGHT, 'yuv420p'
    s.options = {'g': str(gop), 'bf': '0', 'crf': '20', 'preset': 'ultrafast'}
    np.random.seed(idx)
    base = np.random.randint(30, 220, size=3, dtype=np.uint8)
    for i in range(num_frames):
        frame = np.full((HEIGHT, WIDTH, 3), base, dtype=np.uint8)
        frame = np.clip(frame.astype(np.int16) + (i % 30)*3 - 45, 0, 255).astype(np.uint8)
        for pkt in s.encode(av.VideoFrame.from_ndarray(frame, format='rgb24')): c.mux(pkt)
    for pkt in s.encode(): c.mux(pkt)
    c.close()

np.random.seed(42)
lengths = np.random.randint(90, 450, size=NUM_EPISODES)

t0 = time.time()
for idx, length in enumerate(lengths):
    make_episode(EPISODE_DIR / f'episode_{idx:04d}.mp4', length, idx)
    if (idx+1) % 25 == 0: print(f'  {idx+1}/{NUM_EPISODES}...')

total_frames = int(sum(lengths))
total_size = sum(f.stat().st_size for f in EPISODE_DIR.glob('*.mp4'))
print(f'\nDone in {time.time()-t0:.0f}s')
print(f'{NUM_EPISODES} episodes, {total_frames:,} frames, {total_size/(1024*1024):.1f} MB')
print(f'Lengths: min={lengths.min()}, max={lengths.max()}, mean={lengths.mean():.0f}')

---
## 2. Pack into Shards

In [ ]:
from frameforge.packing import PackConfig, pack_shards, ShardDataset, ShardStreamDataset
from frameforge import VideoReader

result = pack_shards(PackConfig(
    episode_paths=sorted(EPISODE_DIR.glob('*.mp4')),
    output_dir=WORK_DIR / 'shards_v1',
    clip_length=16, clips_per_shard=200,
    codec='h264', crf=18, seed=42, backend='pyav',
))
print(result.summary())

# Storage comparison
src_mb = total_size / (1024**2)
shard_mb = result.total_size_bytes / (1024**2)
print(f'\nSource episodes:  {src_mb:.1f} MB')
print(f'Packed shards:    {shard_mb:.1f} MB (GOP=1, all-intra)')
print(f'Ratio:            {shard_mb/src_mb:.2f}x')

### Inspect the manifest

Every clip has full provenance — source episode, exact frame range, shard position.

In [ ]:
manifest = json.loads((WORK_DIR / 'shards_v1' / 'manifest.json').read_text())

print(f'Shards: {len(manifest["shards"])}')
for s in manifest['shards']:
    print(f'  {s["file"]:20s}  {s["num_clips"]:3d} clips  {s["size_bytes"]/1024:.0f} KB')

print(f'\nClips: {len(manifest["clips"])} (first 5):')
for c in manifest['clips'][:5]:
    print(f'  shard {c["shard_id"]} clip {c["clip_index"]:3d}: {c["episode"]} [{c["episode_frame_start"]}:{c["episode_frame_end"]}]')

from collections import Counter
ep_counts = Counter(c['episode'] for c in manifest['clips'])
print(f'\nClips per episode: min={min(ep_counts.values())}, max={max(ep_counts.values())}, mean={sum(ep_counts.values())/len(ep_counts):.1f}')

### Visualize cross-episode shuffling

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

episodes = sorted(set(c['episode'] for c in manifest['clips']))
ep_to_idx = {ep: i for i, ep in enumerate(episodes)}
cmap = plt.cm.tab20

shard_ids = sorted(set(s['shard_id'] for s in manifest['shards']))
fig, axes = plt.subplots(len(shard_ids), 1, figsize=(14, 1.2*len(shard_ids)), sharex=True)
if len(shard_ids) == 1: axes = [axes]

for ax, sid in zip(axes, shard_ids):
    shard_clips = [c for c in manifest['clips'] if c['shard_id'] == sid]
    for ci, clip in enumerate(shard_clips):
        color = cmap(ep_to_idx[clip['episode']] % 20 / 19)
        ax.barh(0, 1, left=ci, height=0.6, color=color, edgecolor='none')
    ax.set_yticks([])
    ax.set_ylabel(f'Shard {sid}', fontsize=8, rotation=0, labelpad=45, ha='right', va='center')
    ax.spines[['top','right','left']].set_visible(False)

axes[-1].set_xlabel('Clip index')
axes[0].set_title('Episode distribution across shards (colors = episodes)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Benchmark — Where the Speedup Comes From

The three costs of reading a clip from an episode file:

| Cost | Direct episode read | Packed shard read |
|------|-------|--------|
| **File open** | Open a new file per clip | File already open (sequential) |
| **Seek** | Find keyframe, decode forward ~15 frames | Zero (GOP=1, already at position) |
| **Decode** | Decode 16 frames | Same |

Let's measure each one separately.

In [ ]:
CLIP_LEN = 16
N = 100
episode_files = sorted(EPISODE_DIR.glob('*.mp4'))

# === Cost 1: File open overhead ===
# Open and close N different episode files (just metadata, no decode)
random.seed(42)
files = [random.choice(episode_files) for _ in range(N)]

t0 = time.perf_counter()
for f in files:
    r = VideoReader(f, backend='pyav')
    _ = len(r)  # forces open
    r.close()
t_open = time.perf_counter() - t0

# === Cost 2: Random seeks within a single file ===
# Keep one file open, seek to N random positions
big_ep = max(episode_files, key=lambda f: f.stat().st_size)
r = VideoReader(big_ep, backend='pyav')
total = len(r)
random.seed(42)
targets = [random.randint(0, total - CLIP_LEN) for _ in range(N)]

t0 = time.perf_counter()
for start in targets:
    _ = r[start:start + CLIP_LEN]
t_seek = time.perf_counter() - t0
r.close()

# === Cost 3: Sequential reads from a single shard ===
shard_ds = ShardStreamDataset(WORK_DIR / 'shards_v1', backend='pyav', prefetch=False)
t0 = time.perf_counter()
for i, clip in enumerate(shard_ds):
    if i >= N: break
t_seq = time.perf_counter() - t0

# === Combined: direct episode reads (open + seek + decode) ===
random.seed(42)
plan = []
for _ in range(N):
    ep = random.choice(episode_files)
    with VideoReader(ep, backend='pyav') as r:
        t = len(r)
    plan.append((ep, random.randint(0, max(0, t - CLIP_LEN))))

t0 = time.perf_counter()
for ep, start in plan:
    with VideoReader(ep, backend='pyav') as r:
        _ = r[start:start + CLIP_LEN]
t_direct = time.perf_counter() - t0

print(f'{N} clips of {CLIP_LEN} frames at {WIDTH}x{HEIGHT}:\n')
print(f'  Cost breakdown:')
print(f'    File opens ({N}x):      {t_open*1000:7.0f} ms  ({t_open/N*1000:.1f} ms each)')
print(f'    Random seeks ({N}x):     {t_seek*1000:7.0f} ms  ({t_seek/N*1000:.1f} ms each)')
print(f'    Sequential shard reads:  {t_seq*1000:7.0f} ms  ({t_seq/N*1000:.1f} ms each)')
print(f'\n  Total comparison:')
print(f'    Direct (open+seek+decode): {t_direct*1000:7.0f} ms  ({t_direct/N*1000:.1f} ms/clip)')
print(f'    Packed sequential:         {t_seq*1000:7.0f} ms  ({t_seq/N*1000:.1f} ms/clip)')
print(f'    Speedup:                   {t_direct/t_seq:.1f}x')
print(f'\n  Where the time goes (direct):')
print(f'    File opens:  ~{t_open/t_direct*100:.0f}%')
print(f'    Seeks:       ~{(t_seek-t_seq)/t_direct*100:.0f}% (seek overhead beyond sequential decode)')
print(f'    Decode:      ~{t_seq/t_direct*100:.0f}% (irreducible decode cost)')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: cost breakdown
labels = ['File opens', 'Seek overhead', 'Decode (sequential)']
vals = [t_open * 1000, (t_seek - t_seq) * 1000, t_seq * 1000]
colors = ['#d95757', '#e2a442', '#3dbea0']
bars = ax1.barh(labels, vals, color=colors, height=0.5)
for bar, v in zip(bars, vals):
    ax1.annotate(f'{v:.0f}ms', xy=(v + max(vals)*0.02, bar.get_y() + bar.get_height()/2),
                va='center', fontsize=9, color='#888')
ax1.set_xlabel('Time (ms)')
ax1.set_title('Where direct reads spend time', fontweight='bold')
ax1.invert_yaxis()

# Right: direct vs packed
methods = ['Direct\n(open+seek+decode)', 'Packed\nsequential']
totals = [t_direct * 1000, t_seq * 1000]
bars2 = ax2.bar(methods, totals, color=['#555555', '#e2a442'], width=0.4)
for bar, t in zip(bars2, totals):
    ax2.annotate(f'{t:.0f}ms', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 5), textcoords='offset points', ha='center', fontsize=9, color='#888')
sp = totals[0]/totals[1] if totals[1] else 0
ax2.annotate(f'{sp:.1f}x', xy=(bars2[1].get_x()+bars2[1].get_width()/2, totals[1]/2),
            ha='center', fontsize=13, fontweight='bold', color='white')
ax2.set_ylabel('Time (ms)')
ax2.set_title(f'{N} clips at {WIDTH}x{HEIGHT}', fontweight='bold')
ax2.grid(axis='y', alpha=0.2); ax2.set_axisbelow(True)

plt.suptitle('Data Packing Benchmark', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Advanced Options

### Codec comparison

In [ ]:
codec_res = {}
for codec in ['h264', 'mjpeg', 'ffv1']:
    r = pack_shards(PackConfig(
        episode_paths=sorted(EPISODE_DIR.glob('*.mp4'))[:20],
        output_dir=WORK_DIR / f'shards_{codec}',
        clip_length=16, clips_per_shard=200,
        codec=codec, crf=18, seed=42, backend='pyav'))
    codec_res[codec] = r

print(f'{"Codec":>8s}  {"Clips":>6s}  {"Size":>10s}  {"Per clip":>10s}  {"Time":>8s}')
print('-' * 50)
for c, r in codec_res.items():
    kb = r.total_size_bytes/1024
    print(f'{c:>8s}  {r.total_clips:6d}  {kb:8.1f}KB  {kb/r.total_clips:8.1f}KB  {r.elapsed_sec:6.1f}s')

### Overlapping clips

In [ ]:
ep20 = sorted(EPISODE_DIR.glob('*.mp4'))[:20]

r_no = pack_shards(PackConfig(episode_paths=ep20, output_dir=WORK_DIR/'nonoverlap',
    clip_length=16, clip_stride=16, clips_per_shard=500, seed=42, backend='pyav'))
r_ov = pack_shards(PackConfig(episode_paths=ep20, output_dir=WORK_DIR/'overlap',
    clip_length=16, clip_stride=4, clips_per_shard=500, seed=42, backend='pyav'))

print(f'Non-overlapping (stride=16): {r_no.total_clips:4d} clips, {r_no.total_size_bytes/1024:.0f} KB')
print(f'Overlapping     (stride=4):  {r_ov.total_clips:4d} clips, {r_ov.total_size_bytes/1024:.0f} KB')
print(f'{r_ov.total_clips/r_no.total_clips:.1f}x more training clips from same source data')

---
## 5. Versioned Re-Packing

In [ ]:
r_v2 = pack_shards(PackConfig(
    episode_paths=sorted(EPISODE_DIR.glob('*.mp4')),
    output_dir=WORK_DIR / 'shards_v2',
    clip_length=32, clips_per_shard=100,
    resolution=(320, 180), seed=99, backend='pyav'))

print('Version comparison:')
for ver, d in [('v1', WORK_DIR/'shards_v1'), ('v2', WORK_DIR/'shards_v2')]:
    m = json.loads((d/'manifest.json').read_text())
    c, s = m['pack_config'], m['stats']
    print(f'  {ver}: clip_len={c["clip_length"]}, res={c.get("resolution","source")}, '
          f'{s["total_clips"]} clips, {s["total_size_bytes"]//1024}KB')

---
## 6. Full Training Pipeline

In [ ]:
import torch
from torch.utils.data import DataLoader

train_ds = ShardStreamDataset(
    WORK_DIR / 'shards_v1', backend='pyav', prefetch=False,
    transform=lambda c: c.float().div(255).permute(0,3,1,2),
    shuffle_shards=True, seed=42)

loader = DataLoader(train_ds, batch_size=8)
for epoch in range(2):
    train_ds.set_epoch(epoch)
    n = 0
    for bi, batch in enumerate(loader):
        n += batch.shape[0]
        if bi == 0: print(f'  Epoch {epoch}: shape={tuple(batch.shape)}, range=[{batch.min():.2f},{batch.max():.2f}]')
    print(f'  Epoch {epoch}: {n} clips')
print('\nDone — feed these to your VLA model.')

---
## Cleanup

In [ ]:
shutil.rmtree(WORK_DIR, ignore_errors=True)
print('Cleaned up.')